<a href="https://colab.research.google.com/github/dgylayse/AkademiQ_DataScience/blob/main/AkademiQ_Data_Science_08.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SVM (Support Vector Machine)

## PARAMETRELER

C = Gerçekte her zaman gruplar kusursuz ayrılmaz. SVM'nin hata toleransına karşılık gelen değerdir.

C büyükse "hiç hata istemiyorum" demiş olursunuz.

C küçükse "birkaç hata olabilir".

c = 0.001-0.1 arası = çok esnek ve model hata yapmaktan çekinmez.

c= 1 başlangıç noktasıdır. Varsayılan değer

c = 10 -100 arasında = katı model. mMdel hata yapsın istemez.

c= 1000 ve üzeri = tehlikeli bölge. Model eğitim verisini ezberler.

C'nin kaç olacağı GridSearchCV ile seçebiliriz.

Kernel trick = Düzlemde ayrılamayan verilerdir. Veri iki boyutlu düzlemde ayrılamıyorsa Kernel fonksiyonlarıyla veriyi daha yüksek bir boyuta taşıyıp, orada ayrılır.

Kernel yüksek boyuttaki koordinatları hesaplaaz noktalar arası benzerlik hesaplar. Bu hesaplama kısayoluna kernel trick denir.

Kernel Türleri

RBF (Radial Basis Function)

Gamma = RBF kerneldeki balonların büyüklüğünü ayarlar. Gamma yüksekse balonlar küçük, model yerel karar verir. Gamma düşükse balonlar büyük. Model daha geniş bölgelere bakarak karar verir.


In [ ]:
import pandas as pd
import numpy as np
import pickle, os
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import (accuracy_score, precision_score, recall_score, roc_auc_score, confusion_matrix)

df = pd.read_csv("musteri_rfm_dt.csv", sep=";")
print(df.head())

  musteri_id  recency_gun  frequency  monetary  ort_sepet  son_islem_ay  \
0  MUS-10418           30          8   9401.71    1175.21            12   
1  MUS-11976           18         14  62669.00    4476.36            12   
2  MUS-12327          127          5   5017.82    1003.56             8   
3  MUS-11241           55          7   8441.74    1205.96            11   
4  MUS-10536           36          3    974.21     324.74            11   

   son_islem_yil dominant_kanal      sehir segment_mevcut  kayip_riski  
0           2024         Mağaza   İstanbul          sadık            0  
1           2024         Online  Gaziantep       şampiyon            0  
2           2024         Online   İstanbul          sadık            0  
3           2024         Mağaza    Antalya          sadık            0  
4           2024         Online     Ankara         riskli            1  


In [ ]:
# EDA (Keşifsel analiz)

for v, cnt in df['kayip_riski'].value_counts().items():
  print(f' {v}  {cnt}  ({cnt/len(df)*100:.1f} %)')

for col in ['recency_gun', 'frequency', 'monetary']:
  print(f"{col:12s}  -> min:{df[col].min():6.0f} max:{df[col].max():6.0f} ort:{df[col].mean():6.0f}")

 0  1200  (50.6 %)
 1  1171  (49.4 %)
recency_gun   -> min:     0 max:   709 ort:   100
frequency     -> min:     1 max:    18 ort:     5
monetary      -> min:     7 max:110871 ort: 12575


In [ ]:
df['kanal_enc'] = LabelEncoder().fit_transform(df['dominant_kanal'])
df['sehir_enc'] = LabelEncoder().fit_transform(df['sehir'])

features = ['recency_gun', 'frequency', 'monetary', 'ort_sepet', 'kanal_enc', 'sehir_enc']
X = df[features]
y = df['kayip_riski']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f'Eğitim:{len(X_train)} satır | Test: {len(X_test)} satır')

Eğitim:1896 satır | Test: 475 satır


In [ ]:
#Standart Scaler (SVM için zorunlu)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc = scaler.transform(X_test)

print("Ölçekleme tamamlandı")
print(f'monetary önce ort: {X_train['monetary'].mean():.0f}  -> sonraki ort {X_train_sc[:,2].mean():.0f}')

Ölçekleme tamamlandı
monetary önce ort: 12464  -> sonraki ort -0


In [ ]:
#SVM Eğitimi

svm = SVC(kernel= 'rbf', C = 1, gamma = 'scale', probability=True)
svm.fit(X_train_sc, y_train)

y_pred = svm.predict(X_test_sc)
y_pred_proba = svm.predict_proba(X_test_sc)[:,1]

from sklearn.metrics import f1_score # Added this import

print(f'Accuracy: {accuracy_score(y_test, y_pred):.4f}')
print(f'Precision: {precision_score(y_test, y_pred):.4f}')
print(f'Recall: {recall_score(y_test, y_pred):.4f}')
print(f'F1: {f1_score(y_test, y_pred):.4f}')
print(f'Roc Auc: {roc_auc_score(y_test, y_pred):.4f}')

Accuracy: 0.9747
Precision: 0.9498
Recall: 1.0000
F1: 0.9742
Roc Auc: 0.9758
